In [ ]:
# goal is to make a flax/jax counterpart of our neural network
# we need to run the same model across multiple seeds on the same gpu on the same batch
# increases GPU memory utilisation, hopefully gives me 10x improvement
# in this notebook though, i simply reimplement a blog to see if the thing even works

# first i do flax basics, then we move ahead
# thike, main main mota mota parts done. now about returning the autoencoder correctly.  
# i would rather only return the encoder weights, the decoder weights and call it a day in singlerun
# first test TX.train though

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
from jax import numpy as jnp, random as jr
from flax import nnx
import optax
import numpy as np
import pandas as pd
import time

In [ ]:
import pt_to_api.benchmark.jax as JX
import pt_to_api.benchmark.torch as TX

# Practice

In [ ]:
class Linear(nnx.Module):
    def __init__(self, din: int, dout: int, *, rngs: nnx.Rngs):
        self.w = nnx.Param(rngs.params.uniform((din, dout)))
        self.b = nnx.Param(jnp.zeros((dout,)))
        self.din, self.dout = din, dout

    def __call__(self, x):
        return x @ self.w + self.b[None]

In [ ]:
r0l = Linear(2, 5, rngs=nnx.Rngs(params=0))
r0l_other = Linear(2, 5, rngs=nnx.Rngs(params=0))

In [ ]:
class Count(nnx.Variable):
    pass

class Counter(nnx.Module):
    def __init__(self):
        self.count = Count(jnp.array(0))
    def __call__(self):
        self.count[...] += 1

In [ ]:
c = Counter()
c.count

In [ ]:
c()
c.count

In [ ]:
class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        self.linear0 = Linear(din, dmid, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, deterministic=False)
        # self.batch_norm = nnx.BatchNorm(dmid, use_running_average=False, rngs=rngs)
        self.linear1 = Linear(dmid, dout, rngs=rngs)

    def __call__(self, x, rngs):
        mid = nnx.gelu(self.dropout(self.linear0(x), rngs=rngs, deterministic=True))
        return self.linear1(mid)

In [ ]:
model = MLP(2, 5, 2, rngs=nnx.Rngs(0))

In [ ]:
model = MLP(2, 16, 10, rngs=nnx.Rngs(0))

optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model: MLP, rngs: nnx.Rngs):
        y_hat = model(x, rngs)
        return jnp.mean((y_hat - y)**2)

    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss

In [ ]:
x, y = jnp.ones((5, 2)), jnp.ones((5, 10))
loss = train_step(model, optimizer, x, y, nnx.Rngs(0))

print(f'{loss = }')
print(f'{optimizer.step.get_value() = }')

## Compare vmap with normal

In [ ]:
class MLP(nnx.Module):
    def __init__(self, din: int, dmid: int, dout: int, *, rngs: nnx.Rngs):
        self.linear0 = Linear(din, dmid, rngs=rngs)
        self.dropout = nnx.Dropout(rate=0.1, deterministic=False)
        # self.batch_norm = nnx.BatchNorm(dmid, use_running_average=False, rngs=rngs)
        self.linear1 = Linear(dmid, dout, rngs=rngs)

    def __call__(self, x, rngs):
        mid = nnx.gelu(self.dropout(self.linear0(x), rngs=rngs, deterministic=True))
        return self.linear1(mid)

@nnx.vmap(in_axes=(0,), out_axes=0)
def create_model(rngs: nnx.Rngs):
    return MLP(2, 10, 2, rngs=rngs)

@nnx.vmap(in_axes=(0,0,None), out_axes=0)
def forward(model: MLP, rngs: nnx.Rngs, x):
    return model(x,rngs)

In [ ]:
key = jax.random.PRNGKey(0)
keys = jax.random.split(key, num=5)  # shape (5,)
stacked_rngs = nnx.Rngs(keys)
ind_rngs = [nnx.Rngs(k) for k in keys]

In [ ]:
x = jnp.ones((1,2))

# model = MLP(2,10,2, rngs=ind_rngs[0])
man_mod, man_out = [], []
for i in range(5):
    model = MLP(2, 10, 2, rngs=ind_rngs[i])
    out = model(x, ind_rngs[i])
    man_mod.append(model)
    man_out.append(out)
man_out

In [ ]:
x = jnp.ones((1, 2))

model = create_model(stacked_rngs)
y = forward(model, stacked_rngs, x)
y

# MNIST

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset

train_steps = 1200
eval_every = 200
batch_size = 32

dataset = load_dataset('ylecun/mnist')
train_ds = dataset['train'].shuffle(seed=0)
test_ds = dataset['test']

def make_batches(ds, batch_size):
  """Yield batches of normalized (image, label) numpy arrays."""
  for i in range(0, len(ds), batch_size):
    batch = ds[i : i + batch_size]
    if len(batch['label']) < batch_size:  # drop incomplete final batch
      break
    images = np.stack([
      np.array(img, dtype=np.float32)[..., None] / 255.0
      for img in batch['image']
    ])
    yield {'image': images, 'label': np.array(batch['label'])}

In [ ]:
batch = next(make_batches(train_ds, 64))

In [ ]:
batch["image"].shape, batch["label"].shape

In [ ]:
from flax import nnx  # The Flax NNX API.
from functools import partial
from typing import Optional

class CNN(nnx.Module):
  """A simple CNN model."""

  def __init__(self, *, rngs: nnx.Rngs):
    self.conv1 = nnx.Conv(1, 32, kernel_size=(3, 3), rngs=rngs)
    self.batch_norm1 = nnx.BatchNorm(32, rngs=rngs)
    self.dropout1 = nnx.Dropout(rate=0.025)
    self.conv2 = nnx.Conv(32, 64, kernel_size=(3, 3), rngs=rngs)
    self.batch_norm2 = nnx.BatchNorm(64, rngs=rngs)
    self.avg_pool = partial(nnx.avg_pool, window_shape=(2, 2), strides=(2, 2))
    self.linear1 = nnx.Linear(3136, 256, rngs=rngs)
    self.dropout2 = nnx.Dropout(rate=0.025)
    self.linear2 = nnx.Linear(256, 10, rngs=rngs)

  def __call__(self, x, rngs: nnx.Rngs | None = None):
    x = self.avg_pool(nnx.relu(self.batch_norm1(self.dropout1(self.conv1(x), rngs=rngs))))
    x = self.avg_pool(nnx.relu(self.batch_norm2(self.conv2(x))))
    x = x.reshape(x.shape[0], -1)  # flatten
    x = nnx.relu(self.dropout2(self.linear1(x), rngs=rngs))
    x = self.linear2(x)
    return x

# Instantiate the model.
model = CNN(rngs=nnx.Rngs(0))
# Visualize it.
nnx.display(model)

In [ ]:
inp = jnp.ones((1, 28, 28, 1))
model(inp, nnx.Rngs(1))

In [ ]:
import optax

lr = 0.005
momentum = 0.9

optimizer = nnx.Optimizer(model, optax.adamw(lr, momentum), wrt=nnx.Param)
metrics = nnx.MultiMetric(
    accuracy=nnx.metrics.Accuracy(),
    loss=nnx.metrics.Average('loss'),
)

In [ ]:
def loss_fn(model: CNN, batch, rngs: nnx.Rngs | None = None):
    logits = model(batch["image"], rngs)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits=logits, labels=batch["label"]).mean()
    return loss, logits


@nnx.jit
def train_step(model, optimizer, metrics, rngs, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch, rngs)
    metrics.update(loss=loss, logits=logits, labels=batch["label"])
    optimizer.update(model, grads)


@nnx.jit
def eval_step(model, metrics, batch):
    loss, logits = loss_fn(model, batch)
    metrics.update(loss=loss, logits=logits, labels=batch["label"])

In [ ]:
model = CNN(rngs=nnx.Rngs(0))
train_model = nnx.view(model, deterministic=False, use_running_average=False)
eval_model = nnx.view(model, deterministic=True, use_running_average=True)

In [ ]:
def pred_step(model, batch):
    logits = model(batch["image"], None)
    return jnp.argmax(logits, axis=1)

def plot_predictions(test_batch, pred):
    fig, axs = plt.subplots(5, 5, figsize=(6, 6))
    for i, ax in enumerate(axs.flatten()):
      ax.imshow(test_batch['image'][i, ..., 0], cmap='binary')
      # ax.set_title(f'label={pred[i]}')
      color = 'green' if test_batch['label'][i] == pred[i] else 'red'
      ax.text(0.05, 0.05, str(pred[i]), transform=ax.transAxes, color=color)
      ax.axis('off')
    return fig

In [ ]:
# %load_ext tensorboard
%tensorboard --logdir runs
# ! tensorboard --logdir runs

In [ ]:
from tensorboardX import SummaryWriter
import tensorboard
writer = SummaryWriter()


In [ ]:
it = enumerate(make_batches(train_ds, batch_size))

In [ ]:
step, batch = next(it)
step, batch["image"].shape

In [ ]:
rngs = nnx.Rngs(0)

In [ ]:
train_step(train_model, optimizer, metrics, nnx.Rngs(0), batch)

In [ ]:
train_step(train_model, optimizer, metrics, nnx.Rngs(0), batch)

In [ ]:
for metric, value in metrics.compute().items():
    writer.add_scalar(f'train_{metric}', value, 0)

In [ ]:
from tqdm import tqdm
rngs = nnx.Rngs(0)

for step, batch in tqdm(enumerate(make_batches(train_ds, batch_size))):
  if step >= train_steps:
    break
  # Run the optimization for one step and make a stateful update to the following:
  # - The train state's model parameters
  # - The optimizer state
  # - The training loss and accuracy batch metrics
  train_step(train_model, optimizer, metrics, rngs, batch)

  if step > 0 and (step % eval_every == 0 or step == train_steps - 1):  # Evaluation period passed.
    # Log the training metrics.
    for metric, value in metrics.compute().items():  # Compute the metrics.
      writer.add_scalar(f'train_{metric}', value, step) # Record the metrics.
    metrics.reset()  # Reset the metrics for the test set.

    # Compute the metrics on the test set after each training epoch.
    for test_batch in make_batches(test_ds, batch_size):
      eval_step(eval_model, metrics, test_batch)

    # Show predicted labels on a single test batch
    pred = pred_step(eval_model, test_batch)
    fig = plot_predictions(test_batch, pred)
    writer.add_figure('inference', fig, step)

    # Log the test metrics.
    for metric, value in metrics.compute().items():
      writer.add_scalar(f'test_{metric}', value, step) # Record the metrics.

    metrics.reset()  # Reset the metrics for the next training epoch.

# Our code

In [ ]:
# import jax
# from pt_to_api.benchmark import train_x as TX
# import torch

# class Autoencoder(nnx.Module):
#     def __init__(self, input_dim, n_components, rngs):
#         self.encoder = nnx.Linear(input_dim, n_components, rngs=rngs, use_bias=False)
#         self.decoder = nnx.Linear(n_components, input_dim, rngs=rngs, use_bias=False)
    
#     def __call__(self, x):
#         codes = self.encoder(x)
#         latent = codes[..., None] * self.decoder.kernel[None, ...]
#         recon = jnp.sum(latent, axis=1)
#         latent_perm = jnp.transpose(latent, (0, 2, 1)) # [batch, columns, components]
#         return recon, codes, latent_perm

In [ ]:
# jax_ae = Autoencoder(10, 4, nnx.Rngs(0))
# torch_ae = TX.Autoencoder(10, 4)
# key = jax.random.PRNGKey(0)
# x = jax.random.normal(key, shape=(1000, 10))
# X_t = torch.tensor(np.array(x))
# x.shape, X_t.shape

In [ ]:
jax_ae.decoder.kernel.shape, torch_ae.decoder.weight.shape

In [ ]:
torch_ae.decoder.weight.shape, jax_ae.decoder.kernel.shape

In [ ]:
jrecon, jcodes, jlatent = jax_ae(x)
jrecon.shape, jcodes.shape, jlatent.shape

In [ ]:
recon, codes, latent = torch_ae(X_t)
recon.shape, codes.shape, latent.shape

In [ ]:
def weights_loss_batched(alpha, sigma_0, W_batch, start_idx=0):
    """Batched version without cyclic shifts. W_batch: [B, C, K]"""
    W_rolled = jnp.roll(W_batch, -start_idx, axis=2)
    W_sq = W_rolled ** 2
    cumsum = jnp.cumsum(W_sq, axis=2)
    phi = alpha * jnp.roll(cumsum, 1, axis=2) + 1
    phi = phi.at[:, :, 0].set(1)
    comp1 = (W_sq * phi / (sigma_0 * sigma_0)).sum(axis=2).mean()
    comp2 = (-jnp.log(phi)).sum(axis=2).mean()
    return comp1, comp2

In [ ]:
jlatent_copy = jnp.array(np.array(latent.detach().numpy()))

In [ ]:
tc1, tc2 = TX.weights_loss_batched(5000, 1, latent)
jc1, jc2 = weights_loss_batched(5000, 1, jlatent_copy)
print("tc1 == jc1", np.allclose(np.array(tc1.detach()), np.array(jc1)))
print("tc2 == jc2", np.allclose(np.array(latent.detach()), np.array(jlatent_copy)))

In [ ]:
((jlatent_copy - 1) ** 2).mean()

In [ ]:
jax.local_devices()

In [ ]:
jax_ae = Autoencoder(10, 4, nnx.Rngs(0))
# torch_ae = TX.Autoencoder(10, 4)
key = jax.random.PRNGKey(0)
x = jax.random.normal(key, shape=(1000, 10))
# X_t = torch.tensor(np.array(x))
x.shape, X_t.shape

In [ ]:
device = jax.local_devices()[0]

In [ ]:
n_components = 4
lr = 1e-2
epochs = 200

x = jax.device_put(x, device)
n_samples, input_dim = x.shape
model = Autoencoder(input_dim, n_components, nnx.Rngs(0))
model = jax.device_put(model, device)
optimizer = nnx.Optimizer(model, optax.adam(lr), wrt=nnx.Param)

In [ ]:
@nnx.jit
def train_step(model, optimizer, batch):
    def loss_fn(model):
        recon, codes, _ = model(batch)
        return recon_loss(batch, recon, 1)
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)

def train_baseline(
    X,
    n_components,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    verbose=True,
):
    print(f"training baseline model, epochs={epochs}")
    X_t = jnp.array(X, dtype=jnp.float32)
    n_samples, input_dim = X_t.shape

    rngs = nnx.Rngs(0)
    model = Autoencoder(input_dim, n_components, rngs=rngs)
    optimizer = nnx.Optimizer(model, optax.adam(lr))

    key = jax.random.PRNGKey(0)

    for epoch in range(epochs):
        key, subkey = jax.random.split(key)
        idx = jax.random.permutation(subkey, n_samples)
        permuted_X_t = X_t[idx]

        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            train_step(model, optimizer, batch)

        if verbose and epoch % 200 == 0:
            recon, codes, _ = model(batch)
            _recon_loss = TX.recon_loss(batch, recon, 1)
            print(f"finetune epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")

    recon, codes, _ = model(X_t)
    components = np.array(model.decoder.kernel)

    return TX.SingleRun(
        model=model,
        codes=np.array(codes),
        components=components,
        recon=np.array(recon),
        loss=float(((X_t - recon) ** 2).mean()),
        hyperparameters={},
    )

In [ ]:
def train_baseline(
    X,
    n_components,
    lr=1e-3,
    epochs=2000,
    batch_size=256,
    verbose=True,
    device="cpu",
):
    """Train the autoencoder with just reconstruction loss, to find an arbitrary linear model which fits the data

    The main training code requires sigma_eps
    the standard deviation of expected gaussian noise
    when the curve is fitted using Y=WX
    We can generally do a simple sweep of hyperparams
    or use simple heuristics
    If the data is linearly "fittable",
    then we get a good starting point
    using this function.
    """
    print(f"training baseline model, epochs={epochs} device={device}")
    jnp.array(X, dtype=jnp.float32)
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    n_samples, input_dim = X_t.shape

    model = Autoencoder(input_dim, n_components)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        idx = torch.randperm(n_samples, device=device)
        permuted_X_t = X_t[idx]
        for i in range(0, n_samples, batch_size):
            batch = permuted_X_t[i : i + batch_size]
            recon, codes, _ = model(batch)
            _recon_loss = recon_loss(batch, recon, 1)
            loss = _recon_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose and epoch % 200 == 0:
            print(f"finetune epoch {epoch:4d} | recon_loss {_recon_loss:.4f}")

    with torch.no_grad():
        recon, codes, _ = model(X_t)

    return TX.SingleRun(
        model.to("cpu"),
        codes.to("cpu").numpy(),
        model.decoder.weight.T.detach().to("cpu").numpy(),
        recon.to("cpu").numpy(),
        ((X_t - recon) ** 2).mean().item(),
        {},
    )

# Test from lib

In [ ]:
# from pt_to_api.benchmark import train_x as TX, train_jax as JX
# from pt_to_api import benchmark as B
from pathlib import Path
import torch
import json

import matplotlib.pyplot as plt
from pt_to_api.utils import show_single_channel_red_green_black as S
SHAPE = (8,9)

In [ ]:

from pt_to_api.benchmark.scalers import NormaliseStdScaler

DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")
MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")

layer_name = "layers.2"
channel = 0

def get_weights_and_patches(layer_data_dir):
    weight = torch.load(layer_data_dir / "weight.pt", weights_only=False)
    patches = torch.load(layer_data_dir / "samples.pt", weights_only=False)
    return weight, patches

def get_loaded_normaliser(file_path):
    normaliser = NormaliseStdScaler()
    state = load_normaliser_state(file_path)
    normaliser.global_std_ = state["global_std_"]
    return normaliser

def load_normaliser_state(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def load_samples_used_for_training(layer_data_dir):
    weight, patches = get_weights_and_patches(layer_data_dir)
    scaler = get_loaded_normaliser(layer_data_dir / "normaliser.json")
    pw = weight * patches
    scaled_pw = scaler.transform(pw)
    return scaled_pw, scaler


In [ ]:
scaled_pw, scaler = load_samples_used_for_training(MAIN_OUT_DIR / layer_name / str(channel))

In [ ]:
data = scaled_pw[:1000]
data.shape

In [ ]:
run = JX.train(data, 4, 5, 1e-2, 400, baseline_epochs=400)

In [ ]:
orig_recon = run[0].recon
orig_codes = run[0].codes

In [ ]:
model = JX.core.autoencoder_from_single_run(run[0])

In [ ]:
new_recon, new_codes, _ = model(jnp.array(data))

In [ ]:
# good, reconstruciton is successful
jnp.all(new_recon == orig_recon), jnp.all(new_codes == orig_codes)